In [1]:
!pip install imbalanced-learn

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE

In [3]:
df = pd.read_csv('/content/fraud.csv')

df.head()

,transaction_amount,hour_of_day,is_weekend,num_items,customer_age,prev_transactions,distance_from_home,device_type,network_quality,is_first_transaction,store_type,velocity_score,is_fraud
0,161.363691,3.0,0.0,2.0,18.000000,2.0,26.539742,1.0,48.403937,0.0,0.0,3.718296,0
1,116.202851,1.0,1.0,4.0,26.285818,2.0,50.714402,NaN,76.144979,0.0,0.0,4.951272,0
2,1.000000,2.0,0.0,5.0,18.000000,NaN,9.467935,0.0,67.600316,0.0,0.0,4.556043,0
3,48.780618,2.0,0.0,3.0,44.471190,NaN,41.077068,0.0,94.825526,0.0,0.0,6.918437,0
4,NaN,3.0,0.0,4.0,38.733609,8.0,NaN,2.0,100.000000,0.0,1.0,5.535335,1


In [4]:
print(df.shape)
print(df.info())

(7000, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   transaction_amount    6440 non-null   float64
 1   hour_of_day           6650 non-null   float64
 2   is_weekend            6860 non-null   float64
 3   num_items             6790 non-null   float64
 4   customer_age          6160 non-null   float64
 5   prev_transactions     6510 non-null   float64
 6   distance_from_home    6300 non-null   float64
 7   device_type           6720 non-null   float64
 8   network_quality       6370 non-null   float64
 9   is_first_transaction  6790 non-null   float64
 10  store_type            6860 non-null   float64
 11  velocity_score        5950 non-null   float64
 12  is_fraud              7000 non-null   int64  
dtypes: float64(12), int64(1)
memory usage: 711.1 KB
None


In [5]:
print(df['is_fraud'].value_counts())

is_fraud
0    6279
1     721
Name: count, dtype: int64


In [6]:
df.fillna(df.median(numeric_only=True), inplace=True)

In [7]:
le = LabelEncoder()

df['device_type'] = le.fit_transform(df['device_type'])
df['store_type'] = le.fit_transform(df['store_type'])

In [8]:
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [11]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print(pd.Series(y_train_smote).value_counts())

is_fraud
0    5023
1    5023
Name: count, dtype: int64


In [12]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_smote, y_train_smote)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

In [13]:
print("Precision:", precision_score(y_test, lr_pred))
print("Recall:", recall_score(y_test, lr_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr_prob))

print(classification_report(y_test, lr_pred))

Precision: 0.10730948678071539
Recall: 0.4791666666666667
ROC-AUC: 0.5211208421797594
              precision    recall  f1-score   support

           0       0.90      0.54      0.68      1256
           1       0.11      0.48      0.18       144

    accuracy                           0.54      1400
   macro avg       0.50      0.51      0.43      1400
weighted avg       0.82      0.54      0.63      1400



In [14]:
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train_smote, y_train_smote)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

In [15]:
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))

print(classification_report(y_test, rf_pred))

Precision: 0.09090909090909091
Recall: 0.013888888888888888
ROC-AUC: 0.5135903220099081
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      1256
           1       0.09      0.01      0.02       144

    accuracy                           0.88      1400
   macro avg       0.49      0.50      0.48      1400
weighted avg       0.81      0.88      0.84      1400



In [16]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1
)

grid_search.fit(X_train_smote, y_train_smote)

print("Best Parameters:", grid_search.best_params_)

Best Parameters: {'max_depth': None, 'n_estimators': 200}


In [17]:
best_rf = grid_search.best_estimator_

best_pred = best_rf.predict(X_test)
best_prob = best_rf.predict_proba(X_test)[:, 1]

print("Precision:", precision_score(y_test, best_pred))
print("Recall:", recall_score(y_test, best_pred))
print("ROC-AUC:", roc_auc_score(y_test, best_prob))

print(classification_report(y_test, best_pred))

Precision: 0.05
Recall: 0.006944444444444444
ROC-AUC: 0.5051364561217269
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      1256
           1       0.05      0.01      0.01       144

    accuracy                           0.88      1400
   macro avg       0.47      0.50      0.48      1400
weighted avg       0.81      0.88      0.84      1400

